## **Problem: Feedback Intelligence System**

#### **Input**
User feedback (text)

#### **Output**
```json
{
  "sentiment": "positive/neutral/negative",
  "category": "product/delivery/support/pricing/other",
  "issue": "short summary of problem"
}
```

## **Architecture**

```
User Input
   ↓
Context Builder
   ↓
Chat Model
   ↓
Raw Response
   ↓
JSON Parser + Validator
   ↓
Final Structured Output
```

## **Building with LangChain**
1. Structured and Reusable Templates
2. Output Parsers
3. Pipiline Abstraction with Chains - Multi-step LLM Calls or Complex pipelines
4. Observability with LangSmith
5. Built-in Integrations out-of-the-box

### **Step 1: Install LangChain**

In [4]:
! pip install langchain

In [5]:
! pip show langchain-core

Name: langchain-core
Version: 1.5.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\krupa\anaconda3\Lib\site-packages
Requires: jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-groq, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk


In [6]:
#pip install langchain
#-langchain-core-premitives like template, output parser
#-langchain-agentic harness
#-langgraph-it is used to build complex custom ai agents and workflows (not covered)
#-langsmith-provides the observability into the ai applications you are building

#used for creating chat model
#pip install langchain-groq
#pip install langchain-cohere
#langchain- (as per there documentation)

### **Step 2: Define Template**

In [8]:
#context builder
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate(
    messages=[
        ("system", "You are a strict JSON generator."),
        ("human", """Analyze the following customer feedback.
        
        Return STRICT JSON with:
        - sentiment (positive, neutral, negative)
        - category (product, delivery, support, pricing, other)
        - issue (short summary)
        
        Feedback:
        ```{feedback}```
        
        Only return JSON. No explanation.
        """)
    ]
)

print(template)

input_variables=['feedback'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a strict JSON generator.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['feedback'], input_types={}, partial_variables={}, template='Analyze the following customer feedback.\n        \n        Return STRICT JSON with:\n        - sentiment (positive, neutral, negative)\n        - category (product, delivery, support, pricing, other)\n        - issue (short summary)\n        \n        Feedback:\n        ```{feedback}```\n        \n        Only return JSON. No explanation.\n        '), additional_kwargs={})]


In [9]:
# Print messages in template
template.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a strict JSON generator.'), additional_kwargs={}),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['feedback'], input_types={}, partial_variables={}, template='Analyze the following customer feedback.\n        \n        Return STRICT JSON with:\n        - sentiment (positive, neutral, negative)\n        - category (product, delivery, support, pricing, other)\n        - issue (short summary)\n        \n        Feedback:\n        ```{feedback}```\n        \n        Only return JSON. No explanation.\n        '), additional_kwargs={})]

In [10]:
# TODO: Calling the template

template.invoke({"feedback":"I like the idea of your product"}) 
#used to replace ```{feedback}```

ChatPromptValue(messages=[SystemMessage(content='You are a strict JSON generator.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Analyze the following customer feedback.\n        \n        Return STRICT JSON with:\n        - sentiment (positive, neutral, negative)\n        - category (product, delivery, support, pricing, other)\n        - issue (short summary)\n        \n        Feedback:\n        ```I like the idea of your product```\n        \n        Only return JSON. No explanation.\n        ', additional_kwargs={}, response_metadata={})])

### **Step 3: Define Chat Model**

In [12]:
! pip install langchain-groq

In [ ]:
from langchain_groq import ChatGroq

with open("keys/.groq_api_key.txt") as f:
    GROQ_API_KEY = f.read()
    
gpt_chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-120b", 
    temperature=1
)

In [14]:
#variable/object name is gpt_chat_model

In [15]:
gpt_chat_model.invoke("hi, how are you!?")

AIMessage(content="Hey there! I'm doing great, thanks for asking. How can I help you today?", additional_kwargs={'reasoning_content': 'The user says "hi, how are you!?" Simple greeting. Should respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 77, 'total_tokens': 123, 'completion_time': 0.096138161, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.013468057, 'prompt_tokens_details': None, 'queue_time': 0.65525329, 'total_time': 0.109606218}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_77b12279f9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00474-f210-7093-b7f7-cb1e7d575e2a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 46, 'total_tokens': 123, 'output_token_details': {'reasoning': 19}})

In [16]:
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

In [17]:
completion = gpt_chat_model.invoke("hi, how are you!?")

completion

AIMessage(content="Hey there! I'm doing great, thanks for asking. How about you? 😊", additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. The user says hi, how are you!? So respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 77, 'total_tokens': 124, 'completion_time': 0.09866993, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.003001987, 'prompt_tokens_details': None, 'queue_time': 0.368294101, 'total_time': 0.101671917}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b1dd3e7a63', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a00474-f6b2-7f90-8c84-51438816af86-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 47, 'total_tokens': 124, 'output_token_details': {'reasoning': 22}})

In [18]:
str_parser.invoke(completion)

"Hey there! I'm doing great, thanks for asking. How about you? 😊"

In [19]:
gpt_chat_model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

In [20]:
## TODO: Calling the Chat Model

### **Step 4: Define Output Schema and Parser**

In [22]:
#pydantic helps us with data validation

In [23]:
from pydantic import BaseModel

class Person(BaseModel):
    name:str
    age:int

In [24]:
person1=Person(name="Krupali", age=20)
print(person1)

name='Krupali' age=20


In [30]:
#person1=Person(name="Krupali", age=twenty)
#print(person1) #validation

In [32]:
from pydantic import BaseModel
from typing import Literal

class FeedbackOutput(BaseModel): #inheritance
    sentiment: Literal["positive", "neutral", "negative"]  #sentiment: - property, type- literal-can't enter any value rather then this 3
    category: Literal["product", "delivery", "support", "pricing", "other"]
    issue: str

In [46]:
f1=FeedbackOutput(
    sentiment="neutral", #sentiment="okish" -> error
    category="product",
    issue="prod is fine"
    )

In [48]:
from langchain_core.output_parsers import PydanticOutputParser

json_parser = PydanticOutputParser(pydantic_object=FeedbackOutput)

### **Step 5: Complete Pipeline**

In [50]:
feedback="i didn't like the delivery at all"

#sentiment-negative,category-delivery,issue-user didn't like the delivery

print(feedback)

i didn't like the delivery at all


In [52]:
context=template.invoke({"feedback":feedback})
print(context)

messages=[SystemMessage(content='You are a strict JSON generator.', additional_kwargs={}, response_metadata={}), HumanMessage(content="Analyze the following customer feedback.\n        \n        Return STRICT JSON with:\n        - sentiment (positive, neutral, negative)\n        - category (product, delivery, support, pricing, other)\n        - issue (short summary)\n        \n        Feedback:\n        ```i didn't like the delivery at all```\n        \n        Only return JSON. No explanation.\n        ", additional_kwargs={}, response_metadata={})]


In [54]:
completion = gpt_chat_model.invoke(context)
print(completion)

content='{\n  "sentiment": "negative",\n  "category": "delivery",\n  "issue": "disliked delivery"\n}' additional_kwargs={'reasoning_content': 'The user wants strict JSON with fields: sentiment, category, issue. Feedback: "i didn\'t like the delivery at all". Sentiment negative. Category delivery. Issue short summary: "disliked delivery" or "unhappy with delivery". Provide JSON.\n\nNeed to ensure strict JSON, no extra whitespace? Probably fine. Output only JSON.\n\n'} response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 150, 'total_tokens': 254, 'completion_time': 0.216329157, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.007723227, 'prompt_tokens_details': None, 'queue_time': 0.271762785, 'total_time': 0.224052384}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_6b677c2caf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0047c-5209-7032-998a-b65dba6223c1-

In [56]:
response=json_parser.invoke(completion)
print(response)

sentiment='negative' category='delivery' issue='disliked delivery'


In [58]:
print(type(response))

<class '__main__.FeedbackOutput'>


In [60]:
response.model_dump()

{'sentiment': 'negative', 'category': 'delivery', 'issue': 'disliked delivery'}

In [62]:
## **MAGIC IMPLEMENTATION**

In [64]:
chain = template | gpt_chat_model | json_parser

In [66]:
fb = "the product is great"

chain.invoke({"feedback":fb})
#feedback is passed to this chain

FeedbackOutput(sentiment='positive', category='product', issue='product is great')